# cli

> Simple CLI for LLM agents — fetch, search, and read commands.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
"""CLI for fossick — fetch, search, and read commands for LLM agents.
Default output is readable markdown; pass --as-json for JSON (piping/harnesses)."""

import json, sys
from fastcore.script import call_parse
from fossick.core import (fetch as _fetch, to_md, read_arxiv as _read_arxiv,
                          read_yt as _read_yt, search_yt as _search_yt,
                          url2nb as _url2nb, download_yt as _download_yt,
                          mv_skill_md, repo_root)
from fossick.search import search as _search, searxng_start as _searxng_start


In [ ]:
#| export
@call_parse
def fetch(
    url:str,              # URL to fetch
    sel:str=None,         # CSS selector to extract (None = full page)
    heavy:bool=False,     # JS rendering via headless browser
    stealthy:bool=False,  # Anti-bot stealth fetcher
    as_json:bool=False,   # Output JSON instead of markdown
):
    "Fetch a URL and print as markdown."
    page = _fetch(url, sel=sel, heavy=heavy, stealthy=stealthy)
    if as_json: print(json.dumps({'url': page['url'], 'status': page['status'], 'content': to_md(page)}))
    else: print(to_md(page))

In [ ]:
#| export
@call_parse
def search(
    query:str,            # search query
    n:int=10,             # number of results
    as_json:bool=False,   # output JSON
):
    "Search the web via SearXNG; prints title, URL, and snippet."
    results = _search(query, n=n)
    if as_json: print(json.dumps([dict(r) for r in results], default=str))
    else:
        for r in results:
            print(f"## {r.title}")
            print(f"URL: {r.url}")
            if r.snippet: print(r.snippet[:200])
            print()

In [ ]:
#| export
@call_parse
def read_arxiv(
    url:str,              # arXiv URL or paper ID
    source:bool=False,    # include full paper text
    chars:int=4000,       # max source chars to include
    as_json:bool=False,   # output JSON
):
    "Fetch arXiv paper metadata, authors, and summary."
    p = _read_arxiv(url)
    if as_json:
        out = {k: v for k, v in p.items() if k != 'source'}
        if source: out['source'] = (p.get('source') or '')[:chars]
        print(json.dumps(out, default=str))
    else:
        print(f"# {p['title']}")
        print(f"Authors: {', '.join(p.get('authors', []))}")
        print(f"Published: {str(p.get('published', ''))[:10]}")
        print(f"\n{p.get('summary', '')}")
        if source and p.get('source'): print(f"\n---\n{p['source'][:chars]}")

In [ ]:
#| export
@call_parse
def read_yt(
    url:str,              # YouTube URL or video ID
    as_json:bool=False,   # output JSON
):
    "Fetch YouTube metadata and full transcript."
    v = _read_yt(url)
    if as_json: print(json.dumps(v, default=str))
    else:
        print(f"# {v['title']}")
        print(f"Channel: {v['channel']}  Duration: {v.get('duration')}s")
        if v.get('source'): print(f"\n{v['source']}")

In [ ]:
#| export
@call_parse
def search_yt(
    query:str,            # search query
    n:int=10,             # number of results
    as_json:bool=False,   # output JSON
):
    "Search YouTube; prints title, URL, and channel."
    results = _search_yt(query, n=n)
    if as_json: print(json.dumps(list(results), default=str))
    else:
        for r in results:
            print(f"## {r['title']}")
            print(f"URL: {r['url']}  Channel: {r.get('channel', '')}  Duration: {r.get('duration')}s")
            if r.get('description'): print(r['description'][:150])
            print()

In [ ]:
#| export
@call_parse
def calls(
    url:str,              # URL to navigate to
    pattern:str='.*',     # regex/glob to filter request URLs
    tail:int=3,           # seconds to wait after navigation
    as_json:bool=False,   # output JSON
):
    "Capture outgoing network requests fired by a page."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect()
        return await cdp.calls(url=url, pattern=pattern, tail=tail)
    caps = syncy(_run())
    if as_json: print(json.dumps(list(caps.values()), default=str))
    else:
        for r in caps.values():
            print(f"## {r['url']}")
            if r.get('response_body'): print(str(r['response_body'])[:300])
            print()


In [ ]:
#| export
@call_parse
def collect(
    url:str,              # URL to open in Chrome
    save_dir:str='.',     # directory to save screenshots
    tout:int=None,        # stop after this many seconds
    count:int=None,       # stop after this many screenshots
    every_n:int=None,     # auto-capture every N seconds
):
    "Open a URL in Chrome and capture screenshots interactively."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect()
        page = await cdp.open_page(url)
        return await page.collect(save_dir=save_dir, tout=tout, count=count, every_n=every_n)
    shots = syncy(_run())
    print(f'{len(shots)} screenshot(s) saved to {save_dir}')


In [ ]:
#| export
@call_parse
def annotate(
    url:str,              # URL to open in Chrome
    save_dir:str='.',     # directory to save annotated screenshot
    as_json:bool=False,   # output JSON list of selected elements
):
    "Open a URL in Chrome, click elements to annotate them, save labeled screenshot."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect()
        page = await cdp.open_page(url)
        return await page.annotate(save_dir=save_dir)
    img, elements = syncy(_run())
    if as_json: print(json.dumps(elements, default=str))
    else:
        for e in elements: print(f"{e['n']}. {e['role']} · {e['selector']}" + (f" — {e['name']}" if e['name'] else ''))


In [ ]:
#| export
@call_parse
def install():
    "Install fossick SKILL.md to .agents/skills/fossick/ and .claude/skills/fossick/."
    mv_skill_md(dry_run=False, dir=repo_root())

In [ ]:
#| export
@call_parse
def url2nb(
    url:str,              # URL to convert (HTML page, PDF, or arXiv)
    path:str=None,        # output notebook path (default: derived from URL)
    as_json:bool=False,   # output JSON with notebook path
):
    "Convert a URL (HTML, PDF, or arXiv) to a Jupyter notebook."
    nb_path = _url2nb(url, nb_path=path)
    if as_json: print(json.dumps({'path': str(nb_path)}))
    else: print(nb_path)


In [ ]:
#| export
@call_parse
def download_yt(
    url:str,              # YouTube URL or video ID
    format:str='audio',   # audio | video | yt-dlp format string
    save_dir:str='.',     # directory to save to
    as_json:bool=False,   # output JSON with saved path
):
    "Download YouTube audio or video."
    path = _download_yt(url, format=format, save_dir=save_dir)
    if as_json: print(json.dumps({'path': str(path)}))
    else: print(path)


In [ ]:
#| export
@call_parse
def start():
    "Start SearXNG and print its URL. Call once per session before using search()."
    url = _searxng_start()
    print(url)

In [ ]:
#| export
CMDS = {
    'fetch':        fetch,
    'search':       search,
    'read-arxiv':   read_arxiv,
    'read-yt':      read_yt,
    'search-yt':    search_yt,
    'url2nb':       url2nb,
    'download-yt':  download_yt,
    'calls':        calls,
    'collect':      collect,
    'annotate':     annotate,
    'install':      install,
    'start':        start,
}

def main():
    "Entry point for the `fossick` CLI command."
    if len(sys.argv) < 2 or sys.argv[1] not in CMDS:
        print(f"Usage: fossick [{'  | '.join(CMDS)}]")
        sys.exit(0 if len(sys.argv) < 2 else 1)
    cmd = sys.argv.pop(1)
    CMDS[cmd]()


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()